In [2]:
import duckdb

# ============================================================
# CONFIGURATION
# ============================================================

DB_PATH = "ecommerce_analytics.duckdb"

PRODUCT_BUY_PATH = r"C:\Users\shyam\Downloads\synerise_dataset\product_buy.parquet"

con = duckdb.connect(DB_PATH)

print("\nConnected to DuckDB.")
print("Building corrected customer segmentation...\n")


# ============================================================
# 1. DATASET REFERENCE DATE
# ============================================================

reference_date = con.execute("""
    SELECT CAST(MAX(last_activity) AS DATE)
    FROM customer_behavior
""").fetchone()[0]

print(f"Dataset reference date: {reference_date}")


# ============================================================
# 2. PURCHASE ACTIVE DAYS
# ============================================================

print("\nCalculating purchase active days...")

con.execute(f"""
CREATE OR REPLACE TABLE purchase_active_days AS

SELECT
    client_id,
    COUNT(DISTINCT CAST(timestamp AS DATE)) AS purchase_active_days

FROM read_parquet('{PRODUCT_BUY_PATH}')

GROUP BY client_id
""")


# ============================================================
# 3. CUSTOMER SEGMENTATION METRICS
# ============================================================

print("Creating customer segmentation metrics...")

con.execute(f"""
CREATE OR REPLACE TABLE customer_segmentation_metrics AS

SELECT

    cb.client_id,

    -- --------------------------------------------------------
    -- Activity
    -- --------------------------------------------------------

    cb.active_days,
    cb.first_activity,
    cb.last_activity,

    -- --------------------------------------------------------
    -- Engagement
    -- --------------------------------------------------------

    cb.page_visits,
    cb.searches,
    cb.cart_additions,
    cb.cart_removals,

    -- --------------------------------------------------------
    -- Product behavior
    -- --------------------------------------------------------

    cb.unique_skus_added,
    cb.unique_skus_removed,

    -- --------------------------------------------------------
    -- Purchase behavior
    -- --------------------------------------------------------

    cb.purchases,
    cb.unique_skus_purchased,
    cb.first_purchase,
    cb.last_purchase,

    COALESCE(
        pad.purchase_active_days,
        0
    ) AS purchase_active_days,

    -- --------------------------------------------------------
    -- Recency
    -- --------------------------------------------------------

    DATE_DIFF(
        'day',
        CAST(cb.last_activity AS DATE),
        DATE '{reference_date}'
    ) AS days_since_last_activity,

    CASE
        WHEN cb.last_purchase IS NOT NULL
        THEN DATE_DIFF(
            'day',
            CAST(cb.last_purchase AS DATE),
            DATE '{reference_date}'
        )
        ELSE NULL
    END AS days_since_last_purchase,

    -- --------------------------------------------------------
    -- Engagement intensity
    -- --------------------------------------------------------

    ROUND(
        cb.page_visits::DOUBLE
        / NULLIF(cb.active_days, 0),
        2
    ) AS page_visits_per_active_day,

    ROUND(
        cb.searches::DOUBLE
        / NULLIF(cb.active_days, 0),
        2
    ) AS searches_per_active_day,

    ROUND(
        cb.cart_additions::DOUBLE
        / NULLIF(cb.active_days, 0),
        2
    ) AS cart_additions_per_active_day,

    ROUND(
        cb.purchases::DOUBLE
        / NULLIF(cb.active_days, 0),
        2
    ) AS purchases_per_active_day,

    -- --------------------------------------------------------
    -- Behavioral ratios
    -- --------------------------------------------------------

    ROUND(
        cb.purchases::DOUBLE
        / NULLIF(cb.cart_additions, 0),
        4
    ) AS purchase_to_cart_ratio,

    ROUND(
        cb.purchases::DOUBLE
        / NULLIF(cb.searches, 0),
        4
    ) AS purchase_to_search_ratio,

    -- --------------------------------------------------------
    -- Total interactions
    -- --------------------------------------------------------

    (
        cb.page_visits
        + cb.searches
        + cb.cart_additions
        + cb.cart_removals
        + cb.purchases
    ) AS total_interactions

FROM customer_behavior cb

LEFT JOIN purchase_active_days pad
    ON cb.client_id = pad.client_id
""")


# ============================================================
# 4. PURCHASER THRESHOLDS
# ============================================================
#
# We use percentile thresholds only for purchasers because
# purchasers have meaningful non-zero behavioral distributions.
#
# Non-purchasers will use explicit meaningful activity levels.

print("Calculating purchaser thresholds...")

thresholds = con.execute("""
WITH purchasers AS (

    SELECT *
    FROM customer_segmentation_metrics
    WHERE purchases > 0

)

SELECT

    quantile_cont(active_days, 0.75)
        AS purchaser_active_days_p75,

    quantile_cont(purchases, 0.75)
        AS purchaser_purchases_p75,

    quantile_cont(
        days_since_last_activity,
        0.75
    ) AS purchaser_recency_p75

FROM purchasers
""").fetchone()


(
    purchaser_active_days_p75,
    purchaser_purchases_p75,
    purchaser_recency_p75
) = thresholds


print("\n========== PURCHASER THRESHOLDS ==========")

print(
    f"Active days P75      : "
    f"{purchaser_active_days_p75:.2f}"
)

print(
    f"Purchases P75        : "
    f"{purchaser_purchases_p75:.2f}"
)

print(
    f"Recency P75          : "
    f"{purchaser_recency_p75:.2f} days"
)


# ============================================================
# 5. CUSTOMER SEGMENTATION
# ============================================================
#
# IMPORTANT:
#
# Non-purchaser rules intentionally use meaningful minimum
# thresholds instead of percentile thresholds.
#
# This avoids the previous problem where:
#
# cart P75    = 0
# search P75  = 0
#
# and therefore almost every customer became "Cart-Heavy".

print("\nAssigning corrected customer segments...")

con.execute(f"""

CREATE OR REPLACE TABLE customer_segmentation AS

SELECT

    *,

    CASE

        -- ====================================================
        -- 1. HIGH-ENGAGEMENT PURCHASER
        -- ====================================================
        -- Strong purchase activity + high engagement + recent
        -- activity relative to other purchasers.

        WHEN purchases > 0

             AND active_days >= {purchaser_active_days_p75}

             AND purchases >= {purchaser_purchases_p75}

             AND days_since_last_activity < {purchaser_recency_p75}

        THEN 'High-Engagement Purchaser'


        -- ====================================================
        -- 2. AT-RISK PURCHASER
        -- ====================================================
        -- Purchased previously but has been inactive for a
        -- relatively long period.

        WHEN purchases > 0

             AND days_since_last_activity >= {purchaser_recency_p75}

        THEN 'At-Risk Purchaser'


        -- ====================================================
        -- 3. REPEAT BUYER
        -- ====================================================
        -- Purchased on at least two distinct days.

        WHEN purchases > 0

             AND purchase_active_days >= 2

        THEN 'Repeat Buyer'


        -- ====================================================
        -- 4. CART-HEAVY NON-PURCHASER
        -- ====================================================
        -- Meaningful cart activity but no purchase.

        WHEN purchases = 0

             AND cart_additions >= 3

        THEN 'Cart-Heavy Non-Purchaser'


        -- ====================================================
        -- 5. SEARCH-HEAVY NON-PURCHASER
        -- ====================================================
        -- Meaningful search activity but no purchase and
        -- not already classified as cart-heavy.

        WHEN purchases = 0

             AND searches >= 3

        THEN 'Search-Heavy Non-Purchaser'


        -- ====================================================
        -- 6. RECENTLY ACTIVE NON-PURCHASER
        -- ====================================================
        -- Recently interacted with the platform but has never
        -- purchased.

        WHEN purchases = 0

             AND days_since_last_activity <= 30

             AND total_interactions > 0

        THEN 'Recently Active Non-Purchaser'


        -- ====================================================
        -- 7. LOW-ENGAGEMENT NON-PURCHASER
        -- ====================================================
        -- Very limited activity.

        WHEN purchases = 0

             AND total_interactions <= 2

        THEN 'Low-Engagement Non-Purchaser'


        -- ====================================================
        -- 8. OTHER PURCHASER
        -- ====================================================

        WHEN purchases > 0

        THEN 'Other Purchaser'


        -- ====================================================
        -- 9. OTHER NON-PURCHASER
        -- ====================================================

        ELSE 'Other Non-Purchaser'

    END AS customer_segment

FROM customer_segmentation_metrics

""")


# ============================================================
# 6. SEGMENT SUMMARY
# ============================================================

print("\n========== CUSTOMER SEGMENT SUMMARY ==========\n")

segment_summary = con.execute("""
SELECT

    customer_segment,

    COUNT(*) AS customers,

    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS customer_percentage,

    ROUND(
        AVG(active_days),
        2
    ) AS avg_active_days,

    ROUND(
        AVG(page_visits),
        2
    ) AS avg_page_visits,

    ROUND(
        AVG(searches),
        2
    ) AS avg_searches,

    ROUND(
        AVG(cart_additions),
        2
    ) AS avg_cart_additions,

    ROUND(
        AVG(purchases),
        2
    ) AS avg_purchases,

    ROUND(
        AVG(unique_skus_purchased),
        2
    ) AS avg_unique_skus_purchased,

    ROUND(
        AVG(days_since_last_activity),
        2
    ) AS avg_days_since_last_activity

FROM customer_segmentation

GROUP BY customer_segment

ORDER BY customers DESC
""").fetchdf()

print(
    segment_summary.to_string(index=False)
)


# ============================================================
# 7. PURCHASE PERFORMANCE BY SEGMENT
# ============================================================

print("\n========== SEGMENT PURCHASE PERFORMANCE ==========\n")

purchase_summary = con.execute("""
SELECT

    customer_segment,

    COUNT(*) AS customers,

    COUNT(*) FILTER (
        WHERE purchases > 0
    ) AS purchasing_customers,

    ROUND(
        COUNT(*) FILTER (
            WHERE purchases > 0
        ) * 100.0 / COUNT(*),
        2
    ) AS purchaser_percentage,

    SUM(purchases) AS purchase_events,

    SUM(cart_additions) AS cart_additions,

    ROUND(
        SUM(purchases) * 1.0
        / NULLIF(SUM(cart_additions), 0),
        4
    ) AS purchase_to_cart_ratio

FROM customer_segmentation

GROUP BY customer_segment

ORDER BY purchase_events DESC
""").fetchdf()

print(
    purchase_summary.to_string(index=False)
)


# ============================================================
# 8. NON-PURCHASER BREAKDOWN
# ============================================================

print("\n========== NON-PURCHASER BREAKDOWN ==========\n")

non_purchaser_summary = con.execute("""
SELECT

    customer_segment,

    COUNT(*) AS customers,

    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_non_purchasers,

    ROUND(
        AVG(page_visits),
        2
    ) AS avg_page_visits,

    ROUND(
        AVG(searches),
        2
    ) AS avg_searches,

    ROUND(
        AVG(cart_additions),
        2
    ) AS avg_cart_additions,

    ROUND(
        AVG(total_interactions),
        2
    ) AS avg_total_interactions,

    ROUND(
        AVG(days_since_last_activity),
        2
    ) AS avg_days_since_last_activity

FROM customer_segmentation

WHERE purchases = 0

GROUP BY customer_segment

ORDER BY customers DESC
""").fetchdf()

print(
    non_purchaser_summary.to_string(index=False)
)


# ============================================================
# 9. VALIDATION CHECK
# ============================================================

print("\n========== VALIDATION ==========\n")

total_customers = con.execute("""
SELECT COUNT(*)
FROM customer_segmentation
""").fetchone()[0]

segmented_customers = con.execute("""
SELECT COUNT(*)
FROM customer_segmentation
WHERE customer_segment IS NOT NULL
""").fetchone()[0]

print(f"Total customers    : {total_customers:,}")
print(f"Segmented customers: {segmented_customers:,}")

if total_customers == segmented_customers:
    print("✓ Every customer has exactly one segment.")
else:
    print("⚠ WARNING: Some customers were not segmented.")


# ============================================================
# 10. FINAL TABLE INFORMATION
# ============================================================

print("\n========== TABLES CREATED ==========\n")

print("✓ purchase_active_days")
print("✓ customer_segmentation_metrics")
print("✓ customer_segmentation")

print("\nCorrected customer segmentation completed successfully.")

con.close()


Connected to DuckDB.
Building corrected customer segmentation...

Dataset reference date: 2022-12-08

Calculating purchase active days...
Creating customer segmentation metrics...
Calculating purchaser thresholds...

========== PURCHASER THRESHOLDS ==========
Active days P75      : 8.00
Purchases P75        : 3.00
Recency P75          : 91.00 days

Assigning corrected customer segments...

========== CUSTOMER SEGMENT SUMMARY ==========

             customer_segment  customers  customer_percentage  avg_active_days  avg_page_visits  avg_searches  avg_cart_additions  avg_purchases  avg_unique_skus_purchased  avg_days_since_last_activity
 Low-Engagement Non-Purchaser    9225684                41.37             1.08             1.21          0.00                0.00           0.00                       0.00                         99.34
          Other Non-Purchaser    5792981                25.98             2.21             8.41          0.08                0.16           0.00          